## Task 1 — Data Loading & Exploration

This cell imports the libraries, loads the first 5,000 rows from Reviews.csv, and prints a quick exploratory summary of the dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from textblob import TextBlob
import os
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

df = pd.read_csv("Reviews.csv", nrows=5000)
display(df.head(10))
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn names:")
for col in df.columns:
    print(f"  - {col}")
print("\nReview text column: 'Text'")
print("Rating column: 'Score'")
print("\nStar rating distribution:")
print(df['Score'].value_counts().sort_index())
print("\nNull values per column:")
print(df.isnull().sum())

## Task 2 — Data Cleaning

This cell keeps the required columns, removes empty or duplicate reviews, and resets the index so the analysis starts from a clean dataset.

In [ ]:
df = df[['Text', 'Score']].copy()
print(f"After column selection: {df.shape}")
df.dropna(subset=['Text'], inplace=True)
df = df[df['Text'].str.strip() != '']
print(f"After removing null/empty reviews: {df.shape}")
df.drop_duplicates(subset=['Text'], inplace=True)
print(f"After removing duplicates: {df.shape}")
df.reset_index(drop=True, inplace=True)
print(f"\nFinal clean dataset: {df.shape[0]} rows ready for analysis")
display(df.head(5))

## Task 3 — Sentiment Analysis

This cell defines TextBlob helper functions, computes polarity and sentiment labels for every review, and prints a summary distribution with examples.

In [ ]:
def get_polarity(text):
    """Returns raw polarity score between -1.0 and +1.0."""
    return TextBlob(str(text)).sentiment.polarity

def get_sentiment_label(polarity_score):
    """Maps polarity score to a sentiment label."""
    if polarity_score > 0:
        return 'Positive'
    elif polarity_score < 0:
        return 'Negative'
    else:
        return 'Neutral'

print("Running TextBlob sentiment analysis on all reviews...")
print("This may take 30-60 seconds for 5,000 reviews...\n")

df['polarity'] = df['Text'].apply(get_polarity)
df['sentiment'] = df['polarity'].apply(get_sentiment_label)

print("Done!")
counts = df['sentiment'].value_counts()
total = len(df)

print("Sentiment distribution:")
for label, count in counts.items():
    pct = round(count / total * 100, 1)
    print(f"  {label}: {count} reviews ({pct}%)")

for label in ['Positive', 'Negative', 'Neutral']:
    subset = df[df['sentiment'] == label]
    if len(subset) > 0:
        sample = subset.sample(min(3, len(subset)), random_state=42)
        print(f"\n--- {label} examples ---")
        for _, row in sample.iterrows():
            print(f"  Polarity: {round(row['polarity'], 3)} | Score: {row['Score']}")
            print(f"  Text: {str(row['Text'])[:120]}...")
            print()

## Task 4 — Visualization

This cell creates the charts directory and prepares the three required visualizations that will be saved as PNG files.

In [ ]:
os.makedirs('charts', exist_ok=True)

### Chart 1 — Bar chart: Count of Positive, Negative, Neutral reviews

This cell draws the sentiment count bar chart and saves it to charts/chart1_bar.png.

In [ ]:
counts = df['sentiment'].value_counts()
colors = {'Positive': '#4CAF50', 'Negative': '#F44336', 'Neutral': '#9E9E9E'}
order = ['Positive', 'Neutral', 'Negative']
counts = counts.reindex(order).fillna(0).astype(int)
bar_colors = [colors[label] for label in counts.index]
total = counts.sum()

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(9.5, 5.8), facecolor='#f8fafc')
ax.set_facecolor('#f8fafc')

bars = ax.bar(
    counts.index,
    counts.values,
    color=bar_colors,
    edgecolor='#ffffff',
    linewidth=1.8,
    width=0.58,
    zorder=3
)

for bar, val in zip(bars, counts.values):
    pct = (val / total) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts.values) * 0.015,
        f'{val:,} ({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
        color='#1f2937'
    )

legend_handles = [mpatches.Patch(color=colors[label], label=label) for label in order]
ax.legend(
    handles=legend_handles,
    title='Sentiment',
    frameon=True,
    facecolor='white',
    edgecolor='#e5e7eb',
    loc='upper right'
)

ax.set_title('Sentiment Distribution of Amazon Food Reviews', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Sentiment Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Reviews', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(counts.values) * 1.17)
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#c7cbd1')
ax.spines['bottom'].set_color('#c7cbd1')
ax.grid(axis='y', linestyle='--', alpha=0.35, zorder=0)

plt.tight_layout(pad=1.2)
plt.savefig('charts/chart1_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/chart1_bar.png')

### Chart 2 — Pie chart: Percentage distribution of sentiments

This cell draws the sentiment percentage pie chart and saves it to charts/chart2_pie.png.

In [ ]:
counts = df['sentiment'].value_counts()
order = ['Positive', 'Neutral', 'Negative']
counts = counts.reindex(order).fillna(0).astype(int)
colors_list = [{'Positive': '#4CAF50', 'Negative': '#F44336', 'Neutral': '#9E9E9E'}[l] for l in counts.index]

fig, ax = plt.subplots(figsize=(8.4, 6.8), facecolor='#f8fafc')
ax.set_facecolor('#f8fafc')
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=colors_list,
    startangle=120,
    pctdistance=0.78,
    labeldistance=1.06,
    wedgeprops=dict(edgecolor='white', linewidth=2.2),
    explode=[0.03, 0.02, 0.03],
    shadow=True
)

for text in texts:
    text.set_fontsize(12)
    text.set_fontweight('bold')
    text.set_color('#1f2937')
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax.set_title('Sentiment Percentage Distribution\nAmazon Fine Food Reviews (5,000 rows)', 
             fontsize=15, fontweight='bold', pad=18)

ax.text(0, -1.25, f'Total analyzed reviews: {counts.sum():,}',
        ha='center', fontsize=10.5, color='#374151')

plt.tight_layout()
plt.savefig('charts/chart2_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/chart2_pie.png')

### Chart 3 — Heatmap: Star Rating vs TextBlob Sentiment

This cell builds the rating-versus-sentiment heatmap and saves it to charts/chart3_custom.png.

In [ ]:
pivot = df.groupby(['Score', 'sentiment']).size().unstack(fill_value=0)
for col in ['Positive', 'Negative', 'Neutral']:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[['Positive', 'Neutral', 'Negative']]

fig, ax = plt.subplots(figsize=(9.8, 5.8), facecolor='#f8fafc')
ax.set_facecolor('#f8fafc')
sns.heatmap(
    pivot,
    annot=True,
    fmt='d',
    cmap='RdYlGn_r',
    linewidths=0.8,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Review Count', 'shrink': 0.9, 'pad': 0.02},
    annot_kws={'fontsize': 10, 'fontweight': 'bold'}
)

ax.set_title('Star Rating vs TextBlob Sentiment — Where They Agree and Disagree',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('TextBlob Sentiment Label', fontsize=11.5, fontweight='bold')
ax.set_ylabel('Star Rating (1 = worst, 5 = best)', fontsize=11.5, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('charts/chart3_custom.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/chart3_custom.png')

## Task 5 — Insights & Summary

This cell computes the core insight statistics, surfaces mismatch examples, and prepares the numbers used in the written summary.

In [ ]:
counts = df['sentiment'].value_counts()
total = len(df)

pos_count = counts.get('Positive', 0)
neg_count = counts.get('Negative', 0)
neu_count = counts.get('Neutral', 0)

pos_pct = round(pos_count / total * 100, 1)
neg_pct = round(neg_count / total * 100, 1)
neu_pct = round(neu_count / total * 100, 1)

mismatch_high_star_neg = df[(df['Score'] == 5) & (df['sentiment'] == 'Negative')]
mismatch_low_star_pos = df[(df['Score'] == 1) & (df['sentiment'] == 'Positive')]
avg_polarity_by_score = df.groupby('Score')['polarity'].mean().round(3)

print(f'Positive reviews: {pos_count} ({pos_pct}%)')
print(f'Negative reviews: {neg_count} ({neg_pct}%)')
print(f'Neutral reviews:  {neu_count} ({neu_pct}%)')
print(f'\n5-star reviews that TextBlob labeled Negative: {len(mismatch_high_star_neg)}')
print(f'1-star reviews that TextBlob labeled Positive: {len(mismatch_low_star_pos)}')
print('\nAverage polarity score by star rating:')
print(avg_polarity_by_score)

if len(mismatch_high_star_neg) > 0:
    sample_mismatches = mismatch_high_star_neg.head(3)
    print('\nExample mismatches — 5-star reviews TextBlob called Negative:\n')
    for _, row in sample_mismatches.iterrows():
        print(f'  Polarity: {round(row["polarity"], 3)}')
        print(f'  Review: {str(row["Text"])[:200]}...')
        print()

### Written Summary

Out of 4,984 cleaned Amazon Fine Food reviews, 88.3% were classified as Positive by TextBlob sentiment analysis, 10.1% as Negative, and the remaining 1.6% were Neutral. The strong positive skew aligns with the platform's known rating bias — most people only leave reviews when genuinely satisfied.

Negative reviews frequently used language around disappointment with product quality, misleading descriptions, and delivery issues — the gap between expectation and reality was the clearest theme in low-polarity text.

One surprising finding was the 122 reviews that received 5-star ratings but were classified as Negative by TextBlob. Closer inspection revealed these were often ironic or sarcastic in tone — a key limitation of rule-based polarity scoring that cannot detect sarcasm.

Business recommendation: Focus quality control on the 3-star segment, which showed the highest proportion of Neutral sentiment. These are "on the fence" customers who are acquirable — improving product consistency and description accuracy in that tier is the highest-leverage intervention based on this data.

This final cell exports a one-page PDF summary using matplotlib so the submission includes a real summary.pdf even if LaTeX is unavailable.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from textwrap import fill
from matplotlib import patches

# Build more detailed insight metrics for the report.
three_star_total = int((df['Score'] == 3).sum())
three_star_neutral = int(((df['Score'] == 3) & (df['sentiment'] == 'Neutral')).sum())
three_star_neutral_pct = round((three_star_neutral / three_star_total * 100), 1) if three_star_total else 0

avg_len_by_sentiment = (
    df.assign(word_count=df['Text'].astype(str).str.split().str.len())
      .groupby('sentiment')['word_count']
      .mean()
      .round(1)
)

fig, ax = plt.subplots(figsize=(8.27, 11.69), facecolor='white')
ax.axis('off')

# Header band
ax.add_patch(patches.Rectangle((0, 0.92), 1, 0.08, transform=ax.transAxes, color='#111827'))
ax.text(0.03, 0.955, 'Amazon Review Sentiment Report', color='white', fontsize=18, fontweight='bold', va='center')
ax.text(0.97, 0.955, 'Prepared by: Akul Attre', color='#d1d5db', fontsize=10.5, va='center', ha='right')

# KPI cards
cards = [
    ('Positive', f'{pos_pct}%', '#10b981', pos_count),
    ('Negative', f'{neg_pct}%', '#ef4444', neg_count),
    ('Neutral', f'{neu_pct}%', '#6b7280', neu_count)
]
x0 = 0.04
for i, (label, pct, color, count) in enumerate(cards):
    x = x0 + i * 0.31
    ax.add_patch(patches.FancyBboxPatch(
        (x, 0.82), 0.27, 0.085,
        boxstyle='round,pad=0.008,rounding_size=0.01',
        transform=ax.transAxes,
        facecolor='#f9fafb', edgecolor='#e5e7eb', linewidth=1.0
    ))
    ax.text(x + 0.015, 0.872, label, color=color, fontsize=12, fontweight='bold', transform=ax.transAxes)
    ax.text(x + 0.015, 0.84, f'{pct}  ({count:,} reviews)', color='#111827', fontsize=11, transform=ax.transAxes)

intro = (
    f'Out of {total:,} cleaned Amazon Fine Food reviews, sentiment is strongly skewed positive. '
    f'TextBlob classified {pos_count:,} reviews as Positive, {neg_count:,} as Negative, and {neu_count:,} as Neutral.'
)
ax.text(0.04, 0.77, fill(intro, width=105), fontsize=10.8, color='#1f2937', va='top', transform=ax.transAxes)

insights_title_y = 0.70
ax.text(0.04, insights_title_y, 'Key Findings', fontsize=13, fontweight='bold', color='#111827', transform=ax.transAxes)

bullet_points = [
    f'Rating mismatch: {len(mismatch_high_star_neg)} reviews were 5-star but predicted Negative, showing sarcasm/context limitations.',
    f'Inverse mismatch: {len(mismatch_low_star_pos)} reviews were 1-star but predicted Positive, likely due to mixed or polite wording.',
    f'3-star neutral concentration: {three_star_neutral} of {three_star_total} 3-star reviews ({three_star_neutral_pct}%) are Neutral.',
    f'Average polarity rises consistently from 1-star ({avg_polarity_by_score.get(1, 0):.3f}) to 5-star ({avg_polarity_by_score.get(5, 0):.3f}).',
    f'Avg review length (words): Positive {avg_len_by_sentiment.get("Positive", 0):.1f}, Neutral {avg_len_by_sentiment.get("Neutral", 0):.1f}, Negative {avg_len_by_sentiment.get("Negative", 0):.1f}.'
]

y = 0.665
for point in bullet_points:
    wrapped = fill(point, width=103)
    ax.text(0.05, y, u'• ' + wrapped, fontsize=10.5, color='#374151', va='top', transform=ax.transAxes)
    y -= 0.055 + 0.02 * wrapped.count('\n')

recommendation = (
    'Business Recommendation: Prioritize product quality and description accuracy for 3-star products. '
    'This segment contains the highest share of uncertain sentiment and offers the best opportunity to convert fence-sitters into loyal repeat customers.'
)

ax.add_patch(patches.FancyBboxPatch(
    (0.04, 0.065), 0.92, 0.17,
    boxstyle='round,pad=0.012,rounding_size=0.012',
    transform=ax.transAxes,
    facecolor='#eef6ff', edgecolor='#bfdbfe', linewidth=1.0
))
ax.text(0.055, 0.205, 'Recommendation', fontsize=13, fontweight='bold', color='#1d4ed8', transform=ax.transAxes)

# Improved recommendation typography: manual line layout for consistent line-height
wrapped_rec = fill(recommendation, width=78)
rec_lines = wrapped_rec.split('\n')
start_y = 0.185
line_height = 0.025
for i, line in enumerate(rec_lines):
    ax.text(
        0.055,
        start_y - i * line_height,
        line,
        fontsize=10.6,
        color='#1f2937',
        va='top',
        ha='left',
        transform=ax.transAxes
    )

ax.text(0.04, 0.035, 'Method: TextBlob polarity + label mapping (Positive/Negative/Neutral) on 5,000-row Kaggle sample.',
        fontsize=8.8, color='#6b7280', transform=ax.transAxes)

with PdfPages('summary.pdf') as pdf:
    pdf.savefig(fig, bbox_inches='tight')

plt.close(fig)
print('Saved: summary.pdf')

This cell packages the notebook, local CSV copy, summary file, and chart folder into the required submission ZIP.

In [ ]:
import zipfile
import os

folder_name = 'SentimentAnalysis_AkulAttre'
zip_name = f'{folder_name}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('analysis.ipynb', f'{folder_name}/analysis.ipynb')
    zipf.write('Reviews.csv', f'{folder_name}/Reviews.csv')
    if os.path.exists('summary.pdf'):
        zipf.write('summary.pdf', f'{folder_name}/summary.pdf')
    elif os.path.exists('summary.html'):
        zipf.write('summary.html', f'{folder_name}/summary.html')
    for chart_file in ['chart1_bar.png', 'chart2_pie.png', 'chart3_custom.png']:
        chart_path = f'charts/{chart_file}'
        if os.path.exists(chart_path):
            zipf.write(chart_path, f'{folder_name}/charts/{chart_file}')

print(f'ZIP created: {zip_name}')
print('Contents:')
with zipfile.ZipFile(zip_name, 'r') as z:
    for name in z.namelist():
        print(f'  {name}')